# 项目一 · Titanic 生存分析

> 五个分析，约 150 行代码。目标不是"跑通"，是**产出一份别人能看懂的报告**。

---

## 题目 1 · 缺失值：统计 + 处理决策

**做什么**：统计每列缺多少、缺的比例；对 Age / Cabin / Embarked 三列**分别给出处理方案和理由**。

**验收标准**：能用一句话说出"为什么 Cabin 用标记、Age 用中位数填、Embarked 直接丢"——
而不是"教程都这么写"。

**状态**：✅ 已完成

---

## 题目 2 · 分布：年龄与票价的直方图

**做什么**：画两个直方图，观察**形状**——是对称？偏斜？有双峰？

**要回答的问题**：
- 年龄分布长什么样？有没有异常（比如大量集中在某个年龄段）？
- 票价分布长什么样？**均值和中间那个人（中位数）差多少？为什么？**

**验收标准**：能说出"Fare 严重右偏，所以**均值不能代表典型乘客**"这类判断。

**用到**：`plt.hist` + `subplots`（第 4 课）

---

## 题目 3 · 关系：相关性热力图

**做什么**：算出数值列之间的相关系数矩阵，画成热力图（记得用发散色系 `RdBu_r`）。

**要回答的问题**：
- 和 `Survived` 关系最强的两个变量是哪两个？
- 哪对变量之间关系最强？这说得通吗？

**验收标准**：能指出最强的相关，**并且说清"相关不等于因果"**——比如舱位和生存率相关，
不代表"买贵票能保命"。

**用到**：`df.corr()` + `imshow`（第 4 课）

---

## 题目 4 · 分组：谁更容易活下来

**做什么**：用 `groupby` 算出 **性别 × 舱位** 的生存率矩阵（一行一个舱位，一列一个性别）。

**要回答的问题**：
- 单看性别，差距多大？单看舱位，差距多大？
- **两者交叉起来看，规律还一样吗？**

**验收标准**：能读出一个"交叉规律"，比如"女性普遍高于男性，而且头等舱的性别差距更大"。

**用到**：`groupby` + `unstack` 或 `pivot_table`（第 3 课）

---

## 题目 5 · 一张能讲故事的图 + 三句话结论

**做什么**：自己设计**一张图**，配**三句话**。

**三句话的格式**（这是这题的重点）：
1. **数据说了什么** —— 一个可验证的事实
2. **这意味着什么** —— 你的解释
3. **你不能从数据推出什么** —— 边界在哪

**验收标准**：别人看这张图 **5 秒**能说出你在讲什么；看那三句话，找不到逻辑漏洞。

**你已经有素材了**：那 15 个"疑似工作人员"的群体画像（全是男性、全在 S 港上船、
上等舱异常多、存活率只有 6.7%）。把它和全船并排对比，就是一张很有说服力的图。

---

## 整体交付标准

| 项 | 要求 |
| :--- | :--- |
| 文件 | `notebooks/project1_titanic_eda.ipynb` |
| 能跑 | 从头 **Run All** 一遍不报错 |
| 图 | 每张都有标题、轴标签、**一条指向重点的注释** |
| 文字 | 每个分析结论写清"数据说了什么"和"不能推出什么" |
| 输出 | 这份 notebook **保留输出**（它是报告，别人点开要能直接看到图） |

---

## 进阶（选做）

同样五个分析，**换成 `data/raw/tmdb_5000_movies.csv` 再做一遍**。

那份数据 4803 行 × 20 列、缺 3941 个值，比 Titanic 脏得多、也真实得多。
同一套方法用在更真实的数据上，说服力完全不同——**这才是能放简历的那一份**。

（提示：TMDB 的 `genres` 列存的是 JSON 字符串，处理方式和普通列不一样，到时候找我。）

In [ ]:

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
df = pd.read_csv("../data/raw/titanic.csv")

# 看形状：多少行多少列 → df.shape
print("形状：")
print(df.shape)
# 看类型和完整性：每列是什么类型、有没有空 → df.info()
print("类型")
print(df.info())
# 看数值分布：范围、均值、中位数、有没有离谱的值 → df.describe()
print("分布")
print(df.describe())
# 看类别取值：每种类别各有多少 → value_counts()
print("取值")
for i in ["Pclass","Sex","Embarked"]:
    print(df[i].value_counts())
# 看缺失：哪列缺、缺多少 → isna().sum()
print("缺失")
print(df[["Age", "Cabin", "Embarked"]].isna().sum())

## 注意到几个问题
1. Age缺失177，Cabin缺失687，Embarked缺失2     需要进行缺失值处理
2. 有人的票价为0，真的有0元的票吗


In [ ]:
#缺失值处理
#Cabin缺失值太多了，已经过半，用一个新的标签来进行标记
df["CabinMiss"] = df["Cabin"].isna()
df["CabinMiss"].value_counts()

#Age缺失177个用中位数进行填充
Age_med = df["Age"].median()
df["Age"] = df["Age"].fillna(Age_med)

#Embarked缺2个直接删除
df = df.dropna(subset=["Embarked"])

## 年龄与票价的直方图

In [ ]:
import matplotlib.pyplot as plt

df1 = pd.read_csv("../data/raw/titanic.csv")
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False  # 不设的话负号会变成方框
plt.rcParams["figure.dpi"] = 110

fig,axes = plt.subplots(1,2,figsize=(10,5))

axes[0].hist(df1["Age"], bins=20, edgecolor="white")
axes[0].set_title("年龄分布图")
axes[0].set_xlabel("年龄")
axes[0].set_ylabel("人数")

axes[1].hist(df1["Fare"], bins=20, edgecolor="white")
axes[1].set_title("票价分布图")
axes[1].set_xlabel("票价")
axes[1].set_ylabel("人数")
#年龄大量集中在20-24岁左右，票价分布则右偏严重，均值和中位数相差较大

## 相关热力图

In [ ]:
corr = df[["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare"]].corr()

fig,axes = plt.subplots(figsize=(8,6))
im = axes.imshow(corr.values, cmap="RdBu_r",vmin=-1,vmax=1)
fig.colorbar(im, ax=axes,label = "相关系数")
axes.set_xticks(range(len(corr)),corr.columns,rotation=45)
axes.set_yticks(range(len(corr)),corr.columns,)
for i in range(len(corr)):
    for j in range(len(corr)):
        axes.text(j, i, corr.values[i, j].round(2), ha="center", va="center",
                  fontsize=12,
                  color="white" if abs(corr.values[i, j]) > 0.5 else "black")
# 和 Survived 关系最强的是 Pclass（-0.34，负相关）和 Fare（+0.26，正相关）
# SibSp 和 Parch 都接近 0，几乎没有关系
# 相关不代表因果：舱位高的人生存率高，不代表"买贵票能保命"

## 谁更容易活下来

In [ ]:
pd.pivot_table(df, values="Survived", index="Sex", columns="Pclass", aggfunc="mean", margins=True, margins_name="全船")
#单看船舱，1等舱的存活率最高，2等舱次之，3等舱最低
#单看性别，男性的存活率要低于女性
#一等舱的女性更容易活下来

## 搞清楚
1. 票价为0的人的信息

In [ ]:
#他们的性别是什么
df[df["Fare"] == 0].groupby(["Sex"]).size()
#发现票价为0的人都是男性

In [ ]:
# 他们最后的存活率怎么样
df[df["Fare"] == 0]["Survived"].mean().round(2)
#存活率居然只有0.07，几乎无人生还
#和全船的人进行对比一下
df["Survived"].mean().round(2)
#全船的存货率也只有0.38，但还是比他们的存活率高出非常多

In [ ]:
#他们的船舱等级
df[df["Fare"] == 0].groupby(["Pclass"]).size()
#发现他们几乎均匀分布在三个等级的船舱里

In [ ]:
#他们的登船港口
df[df["Fare"] == 0].groupby(["Embarked"]).size()
#发现都是从s港登船的


In [ ]:
#他们的客舱编号
df[df["Fare"] == 0].groupby("CabinMiss").size()
#发现他们15人中只有3个人是有客舱的，其余12个人并没有
df[df["Fare"] == 0]["Cabin"].value_counts()
#3人分别在B94 A36 B102

In [ ]:
def profile(data):
    """输入一群人，输出四个指标的比例。"""
    return {
        "男性占比": (data["Sex"] == "male").mean(),
        "上等舱占比": data["Pclass"].isin([1, 2]).mean(),
        "S 港上船占比": (data["Embarked"] == "S").mean(),
        "存活率": data["Survived"].mean(),
    }


group = profile(df[df["Fare"] == 0])
all_person = profile(df)

contrast = pd.DataFrame({"这 15 人": group, "全船": all_person})
print(contrast.round(3))


In [ ]:
trait = contrast.index.tolist()
y = np.arange(len(trait))
width = 0.38

fig, ax = plt.subplots(figsize=(9.5, 5))

ax.barh(y - width / 2, contrast["这 15 人"], height=width,
        label="这 15 人（票价 = 0）", color="#C44E52")
ax.barh(y + width / 2, contrast["全船"], height=width,
        label="全船 891 人", color="#4C72B0")

ax.set_yticks(y, trait)
ax.set_xlim(0, 1.12)
ax.set_xlabel("比例")
ax.set_title("那 15 个「票价 0」的人，和全船哪里不一样", fontsize=13, pad=12)
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)

# 把数值写在柱子末端
for i, (a, b) in enumerate(zip(contrast["这 15 人"], contrast["全船"])):
    ax.text(a + 0.015, i - width / 2, f"{a:.1%}", va="center", fontsize=10)
    ax.text(b + 0.015, i + width / 2, f"{b:.1%}", va="center", fontsize=10, color="dimgray")

# 一条注释指向重点
ax.annotate("这 15 人的存活率只有全船的 1/6",
            xy=(0.067, 3 - width / 2), xytext=(0.3, 3.4),
            arrowprops=dict(arrowstyle="->", color="gray"),
            fontsize=10, color="dimgray")

fig.tight_layout()
plt.show()

## 我发现的：那 15 个「票价 0」的人

**① 数据说了什么**

在 891 名乘客里，有 15 个人的票价为 0。他们和全船在四个指标上都明显不同：
全是男性（全船 64.8%）、上等舱占 73%（全船 44.9%）、全部从 S 港上船（全船 72.3%）、
存活率只有 6.7%（全船的 1/6）。

**② 这意味着什么**

这个画像不像买票的普通乘客——他们更像是**随船的工作人员**：有登记、有上等舱位、
没有家庭、全部男性，且几乎全部遇难。

**③ 我不能从数据推出什么**

数据只能证明「他们和其他人不一样」，**不能证明「他们是谁」**。要确认身份需要额外资料。
另外三点也要说清：票价 0 不等于免费票，也可能只是记账方式；
15 个人的样本太小，「全是男性」也可能只是巧合；
而「票价 0」和「几乎全遇难」哪个是因、哪个是果，从这份数据里看不出来。